In [ ]:
!pip install -q kailash kailash-ml polars torch torchvision matplotlib python-dotenv torch-geometric


# MLFP05 — Exercise 6.4: Link Prediction with GNN Encoder-Decoder


**WHAT YOU'LL LEARN:**
- Why link prediction matters (knowledge graphs, social networks, recs)
- Encoder-decoder architecture: GNN encoder + dot-product decoder
- Positive vs negative edge sampling for training
- AUC metric for ranking quality evaluation
- Train a link predictor on the Cora citation network
- Track training with kailash-ml ExperimentTracker


**PREREQUISITES: M5/ex_6.1 (GCN layer implementation).**

**ESTIMATED TIME: ~30 min**


In [ ]:
from __future__ import annotations

import asyncio

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from shared.mlfp05.ex_6 import (
    OUTPUT_DIR,
    device,
    load_graph_data,
    plot_training_curves,
    register_model,
    setup_engines,
)
from kailash_ml.types import MetricSpec

import matplotlib.pyplot as plt


## PHASE 1 — THEORY: Why Link Prediction Matters
Node classification asks "what is this node?" Link prediction asks
"should these two nodes be connected?" This is the foundation of:
1. KNOWLEDGE GRAPHS: A medical knowledge graph has nodes for drugs,
diseases, proteins, and genes. Edges represent known interactions
(drug X treats disease Y). Link prediction discovers MISSING edges
— potential new drug-disease interactions that haven't been tested.
2. SOCIAL NETWORKS: "People you may know" = predict missing friendship
edges based on mutual connections and profile features.
3. RECOMMENDATION: "Papers you should cite" = predict missing citation
edges based on content similarity and citation patterns.
The approach: ENCODER-DECODER architecture
- ENCODER: A GNN (like our GCN) that produces node embeddings z_i
- DECODER: dot-product similarity between embeddings
score(i, j) = sigmoid( z_i^T z_j )
If two nodes have similar embeddings, the decoder predicts an edge
between them. Training uses known edges as positives and random
non-edges as negatives — binary classification on edge existence.


In [ ]:
print("=" * 70)
print("  PHASE 1 — THEORY: Link Prediction on Graphs")
print("=" * 70)
print(
    """
  LINK PREDICTION: "Should these two nodes be connected?"

  THREE MAJOR APPLICATIONS:
  1. Knowledge Graphs: discover new drug-disease interactions
  2. Social Networks: "people you may know" suggestions
  3. Citation Networks: "papers you should cite" recommendations

  ENCODER-DECODER APPROACH:
  - ENCODER (GNN): learns node embeddings z_i from features + structure
  - DECODER (dot product): score(i,j) = sigmoid(z_i^T z_j)
  - High similarity in embedding space -> predict edge exists

  TRAINING DATA:
  - Positive samples: real edges from the graph (label = 1)
  - Negative samples: random non-edges (label = 0)
  - Loss: binary cross-entropy on edge predictions
  - Metric: AUC — how well do we rank real edges above non-edges?
"""
)


## PHASE 2 — BUILD: GNN Encoder + Dot-Product Decoder


In [ ]:
print("=" * 70)
print("  PHASE 2 — BUILD: Link Prediction Model")
print("=" * 70)

# Load graph data and set up engines
graph_data = load_graph_data()
conn, tracker, exp_name, registry, has_registry = setup_engines()

X = graph_data["X"]
A = graph_data["A"]
A_norm = graph_data["A_norm"]
A_np = graph_data["A_np"]
y_np = graph_data["y_np"]
edge_index_np = graph_data["edge_index_np"]
N = graph_data["N"]
F_dim = graph_data["F_dim"]
n_classes = graph_data["n_classes"]
dataset_name = graph_data["dataset_name"]

HIDDEN_DIM = 16 if dataset_name == "Karate Club" else 64
LINK_EPOCHS = 80


# Reuse GCN layer for the encoder
class GCNLayer(nn.Module):
    """GCN layer: H' = A_norm @ (H @ W)."""

    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=True)

    def forward(self, h: torch.Tensor, a_norm: torch.Tensor) -> torch.Tensor:
        return a_norm @ self.W(h)


class LinkPredictor(nn.Module):
    """Encoder-decoder for link prediction.

    Encoder: MLP -> two GCN layers that produce node embeddings.
    Decoder: dot product between node pairs -> edge probability.

    The encoder first projects features through an MLP (to handle
    high-dimensional bag-of-words features), then applies two GCN
    layers to incorporate graph structure into the embeddings.
    """

    def __init__(self, in_dim: int, hidden_dim: int):
        super().__init__()
        # TODO: Build the encoder — an MLP followed by two GCN layers:
        # 1. self.encoder = nn.Sequential(
        #        nn.Linear(in_dim, hidden_dim),
        #        nn.ReLU(),
        #        nn.Linear(hidden_dim, hidden_dim),
        #    )
        # 2. self.gcn1 = GCNLayer(hidden_dim, hidden_dim)
        # 3. self.gcn2 = GCNLayer(hidden_dim, hidden_dim)
        pass

    def encode(self, h: torch.Tensor, a_norm: torch.Tensor) -> torch.Tensor:
        """Produce node embeddings: MLP -> GCN -> GCN."""
        # TODO: Pass h through encoder MLP, then GCN1 with ReLU, then GCN2
        # h = self.encoder(h)
        # h = F.relu(self.gcn1(h, a_norm))
        # h = self.gcn2(h, a_norm)
        # return h
        pass

    def decode(
        self, z: torch.Tensor, src: torch.Tensor, dst: torch.Tensor
    ) -> torch.Tensor:
        """Dot-product decoder: score(i,j) = z_i^T z_j."""
        # TODO: Compute dot product between source and destination embeddings
        # Hint: return (z[src] * z[dst]).sum(dim=-1)
        pass

    def forward(
        self,
        h: torch.Tensor,
        a_norm: torch.Tensor,
        src: torch.Tensor,
        dst: torch.Tensor,
    ) -> torch.Tensor:
        z = self.encode(h, a_norm)
        return self.decode(z, src, dst)


link_model = LinkPredictor(F_dim, HIDDEN_DIM).to(device)
n_params = sum(p.numel() for p in link_model.parameters())

print(f"\n  Link Prediction architecture:")
print(
    f"    Encoder MLP: Linear({F_dim}->{HIDDEN_DIM}) -> ReLU -> Linear({HIDDEN_DIM}->{HIDDEN_DIM})"
)
print(f"    GCN Layer 1: GCNLayer({HIDDEN_DIM} -> {HIDDEN_DIM})")
print(f"    GCN Layer 2: GCNLayer({HIDDEN_DIM} -> {HIDDEN_DIM})")
print(f"    Decoder: dot_product(z_i, z_j) -> edge score")
print(f"    Total parameters: {n_params:,}")


### Build Checkpoint ────────────────────────────────────────────────


In [ ]:
assert isinstance(link_model, nn.Module), "LinkPredictor should be an nn.Module"
assert n_params > 0, "LinkPredictor should have learnable parameters"
print("\n--- Build checkpoint passed --- LinkPredictor architecture created\n")


## PHASE 3 — TRAIN: Link Prediction on Cora


In [ ]:
print("=" * 70)
print(f"  PHASE 3 — TRAIN: Link Prediction on {dataset_name}")
print("=" * 70)

# Prepare positive and negative edge samples
# Positive: real edges from the graph
pos_src = torch.from_numpy(edge_index_np[0]).to(device)
pos_dst = torch.from_numpy(edge_index_np[1]).to(device)
n_pos = len(pos_src)

# TODO: Sample negative edges — random node pairs that are NOT connected
# 1. Use rng = np.random.default_rng(42)
# 2. Repeatedly sample random (s, d) pairs where s != d and A_np[s, d] == 0
# 3. Collect n_pos negative edges to match the positive count
# 4. Convert to tensors on device
# Hint: while neg_count < n_pos:
#          s = rng_link.integers(0, N); d = rng_link.integers(0, N)
#          if s != d and A_np[s, d] == 0: add to lists
neg_src_list = []
neg_dst_list = []
rng_link = np.random.default_rng(42)
# TODO: Fill neg_src_list and neg_dst_list
neg_count = 0
# while neg_count < n_pos:
#     s = rng_link.integers(0, N)
#     d = rng_link.integers(0, N)
#     if s != d and A_np[s, d] == 0:
#         neg_src_list.append(s)
#         neg_dst_list.append(d)
#         neg_count += 1
neg_src = torch.tensor(neg_src_list, dtype=torch.long, device=device)
neg_dst = torch.tensor(neg_dst_list, dtype=torch.long, device=device)

print(f"  Positive edges (real connections): {n_pos:,}")
print(f"  Negative edges (random non-edges): {len(neg_src):,}")
print(f"  Ratio: 1:1 (balanced)")

# Train
link_opt = torch.optim.Adam(link_model.parameters(), lr=1e-2, weight_decay=1e-4)
link_losses: list[float] = []
link_aucs: list[float] = []


async def _train_link_predictor_async():
    """Train the link predictor under a tracker.run(...) context."""
    async with tracker.run(experiment_name=exp_name, run_name="link_prediction") as ctx:
        await ctx.log_params(
            {
                "task": "link_prediction",
                "hidden_dim": str(HIDDEN_DIM),
                "epochs": str(LINK_EPOCHS),
                "n_pos_edges": str(n_pos),
                "n_neg_edges": str(len(neg_src)),
            }
        )

        for epoch in range(LINK_EPOCHS):
            link_model.train()
            link_opt.zero_grad()

            # TODO: Compute positive and negative scores, then BCE loss
            # 1. pos_scores = link_model(X, A_norm, pos_src, pos_dst)
            # 2. neg_scores = link_model(X, A_norm, neg_src, neg_dst)
            # 3. Concatenate scores and labels:
            #    scores = torch.cat([pos_scores, neg_scores])
            #    labels = torch.cat([ones for pos, zeros for neg])
            # 4. loss = F.binary_cross_entropy_with_logits(scores, labels)
            # 5. loss.backward(); link_opt.step()
            # Hint: torch.ones(n_pos, device=device) for positive labels

            # TODO: Compute AUC-like metric
            # link_model.eval()
            # with torch.no_grad():
            #     p_scores = link_model(X, A_norm, pos_src, pos_dst)
            #     n_scores = link_model(X, A_norm, neg_src, neg_dst)
            #     n_sample = min(1000, n_pos, len(neg_src))
            #     auc_approx = (p_scores[:n_sample] > n_scores[:n_sample]).float().mean().item()
            # link_aucs.append(auc_approx)

            # TODO: Log metrics
            # await ctx.log_metrics({"link_loss": loss.item(), "link_auc_approx": auc_approx}, step=epoch+1)

            if (epoch + 1) % 20 == 0:
                print(
                    f"  [LinkPred] epoch {epoch+1:3d}  "
                    f"loss={link_losses[-1] if link_losses else 0:.4f}  "
                    f"auc_approx={link_aucs[-1] if link_aucs else 0:.3f}"
                )

        if link_losses and link_aucs:
            await ctx.log_metrics(
                {
                    "final_link_loss": link_losses[-1],
                    "final_link_auc": link_aucs[-1],
                }
            )


await _train_link_predictor_async()


### Train Checkpoint ────────────────────────────────────────────────


In [ ]:
assert len(link_losses) == LINK_EPOCHS, "Link prediction should train for all epochs"
assert link_losses[-1] < link_losses[0], "Link prediction loss should decrease"
assert (
    link_aucs[-1] > 0.55
), f"Link prediction AUC {link_aucs[-1]:.3f} should exceed random (0.5)"
final_auc = link_aucs[-1]
print(f"\n  Link Prediction Results:")
print(f"    Final loss:      {link_losses[-1]:.4f}")
print(f"    Final AUC:       {final_auc:.4f}")
print(f"    Best AUC:        {max(link_aucs):.4f}")
print(f"    Random baseline: 0.5000")
# INTERPRETATION: The link predictor learns that connected papers have
# similar GNN embeddings. The dot-product decoder measures embedding
# similarity — high similarity predicts a citation link. An AUC > 0.5
# means the model ranks real edges higher than random non-edges.
print("\n--- Train checkpoint passed --- link prediction trained\n")


## PHASE 4 — VISUALISE: Edge Scores + Embedding Similarity


In [ ]:
print("=" * 70)
print("  PHASE 4 — VISUALISE: Link Prediction Analysis")
print("=" * 70)

link_model.eval()
with torch.no_grad():
    # Get node embeddings
    z = link_model.encode(X, A_norm).cpu().numpy()

    # Score distributions for positive vs negative edges
    pos_final_scores = link_model(X, A_norm, pos_src, pos_dst).cpu().numpy()
    neg_final_scores = link_model(X, A_norm, neg_src, neg_dst).cpu().numpy()

# TODO: Create visualisation with 2 subplots:
# Left: Histogram of score distributions (positive vs negative edges)
#   - pos_final_scores[:2000] in green with label "Real edges"
#   - neg_final_scores[:2000] in red with label "Non-edges"
#   - density=True, bins=60
# Right: Training progress (loss and AUC on twin y-axes)
#   - Loss on left y-axis in steelblue
#   - AUC on right y-axis in coral (use ax.twinx())
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# TODO: Fill in the histogram and training curve plots
# axes[0].hist(pos_final_scores[:2000], bins=60, alpha=0.7, color="green", ...)
# axes[0].hist(neg_final_scores[:2000], bins=60, alpha=0.7, color="red", ...)

fig.tight_layout()
filepath = OUTPUT_DIR / "link_prediction_analysis.png"
fig.savefig(filepath, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  Saved: {filepath}")

# TODO: Create heatmap comparing true adjacency vs predicted similarity
# 1. Select n_sub=50 random nodes
# 2. Compute dot-product similarity: z_sub @ z_sub.T
# 3. Plot side-by-side: true adjacency matrix vs similarity matrix
# Hint: fig, axes = plt.subplots(1, 2, figsize=(14, 6))
#        axes[0].imshow(sub_A, cmap="Blues")
#        axes[1].imshow(similarity, cmap="RdBu_r")
n_sub = min(50, N)
rng = np.random.default_rng(42)
sub_idx = rng.choice(N, n_sub, replace=False)
sub_idx = np.sort(sub_idx)
z_sub = z[sub_idx]
similarity = z_sub @ z_sub.T  # dot-product similarity
sub_A = A_np[np.ix_(sub_idx, sub_idx)]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# TODO: Plot true adjacency and predicted similarity side-by-side

fig.suptitle(
    f"Link Prediction: True Edges vs Learned Similarities — {dataset_name}",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
filepath = OUTPUT_DIR / "link_prediction_similarity.png"
fig.savefig(filepath, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  Saved: {filepath}")

# Training curves via ModelVisualizer
plot_training_curves(
    metrics_dict={
        "Link pred loss": link_losses,
        "Link pred AUC (approx)": link_aucs,
    },
    title="Link Prediction Training",
    y_label="Value",
    filename="link_prediction_curves.html",
)

# Score statistics
pos_mean = pos_final_scores.mean()
neg_mean = neg_final_scores.mean()
print(f"\n  Score analysis:")
print(f"    Real edges   — mean score: {pos_mean:+.4f}")
print(f"    Non-edges    — mean score: {neg_mean:+.4f}")
print(f"    Separation:  {pos_mean - neg_mean:.4f}")
print(f"    -> Good separation means the model can distinguish real from fake edges")


### Visualise Checkpoint ────────────────────────────────────────────


In [ ]:
assert z.shape == (N, HIDDEN_DIM), f"Embedding shape should be ({N}, {HIDDEN_DIM})"
assert pos_mean > neg_mean, "Positive edges should score higher on average"
print("\n--- Visualise checkpoint passed --- link prediction analysis plotted\n")


## PHASE 5 — APPLY: Knowledge Graph Completion for a Hospital


In [ ]:
print("=" * 70)
print("  PHASE 5 — APPLY: Knowledge Graph Completion at SGH")
print("=" * 70)
print(
    """
  SCENARIO: You're building a drug-disease interaction predictor for
  Singapore General Hospital (SGH) using a medical knowledge graph.

  THE KNOWLEDGE GRAPH:
  - Drug nodes: ~5K approved drugs (features: molecular weight, targets, ATC code)
  - Disease nodes: ~10K conditions (features: ICD-10 code, organ system, prevalence)
  - Protein nodes: ~20K proteins (features: function, pathway, expression level)
  - Edges: known interactions (drug-treats-disease, drug-binds-protein,
    protein-associated-with-disease)

  LINK PREDICTION TASK: Discover new drug-disease edges
  - Known: drug X treats disease Y (from clinical trials)
  - Unknown: does drug X also treat disease Z? (drug repurposing)
  - Validation: withhold 20% of known edges, predict them

  HOW IT WORKS:
  1. Encode all nodes with GNN: each drug gets an embedding that
     captures its molecular features AND its known interactions
  2. Score all (drug, disease) pairs with dot product
  3. Rank by score — top-k are candidate interactions for lab testing
  4. Validate: do withheld known interactions appear in top-k?
"""
)

# TODO: Demonstrate edge rediscovery experiment
# 1. Withhold 100 real edges (randomly selected)
# 2. Score withheld edges with the trained model
# 3. Score random non-edges for comparison
# 4. Compute rediscovery AUC: fraction of withheld edges scoring above random
# Hint: withheld_scores = link_model(X, A_norm, withheld_src, withheld_dst)
#        random_scores = link_model(X, A_norm, rand_src, rand_dst)
#        rediscovery_auc = (withheld_scores > random_scores).float().mean().item()
print("  Demonstration: edge rediscovery experiment")
print("  (Withholding 100 real edges, checking if the model scores them highly)\n")

rng_demo = np.random.default_rng(123)
n_test_edges = min(100, n_pos)
test_edge_indices = rng_demo.choice(n_pos, n_test_edges, replace=False)

withheld_src = pos_src[test_edge_indices]
withheld_dst = pos_dst[test_edge_indices]

# TODO: Score withheld edges and compare to random non-edges
# with torch.no_grad():
#     withheld_scores = link_model(X, A_norm, withheld_src, withheld_dst)
#     rand_src = torch.randint(0, N, (n_test_edges,), device=device)
#     rand_dst = torch.randint(0, N, (n_test_edges,), device=device)
#     random_scores = link_model(X, A_norm, rand_src, rand_dst)
#
# withheld_mean = withheld_scores.mean().item()
# random_mean = random_scores.mean().item()
# rediscovery_auc = (withheld_scores > random_scores).float().mean().item()

withheld_mean = 0.0  # Replace with your computed value
random_mean = 0.0  # Replace with your computed value
rediscovery_auc = 0.0  # Replace with your computed value

print(f"    Withheld real edges — mean score: {withheld_mean:+.4f}")
print(f"    Random non-edges   — mean score: {random_mean:+.4f}")
print(f"    Rediscovery AUC:                  {rediscovery_auc:.4f}")
print(f"    (1.0 = perfectly ranks real edges above non-edges)")

if rediscovery_auc > 0.6:
    print("    -> Model successfully rediscovers withheld edges!")
    print(
        "    -> In a hospital setting: these would be drug-disease candidates for trials"
    )

print(
    """
  CLINICAL DEPLOYMENT:
  1. Build KG from DrugBank, OMIM, STRING databases + SGH clinical records
  2. Train link predictor on known drug-disease edges
  3. Score all (drug, disease) pairs without known interactions
  4. Top-k candidates reviewed by pharmacology team for literature evidence
  5. Promising candidates enter pre-clinical or retrospective cohort studies
  6. Track predictions with ExperimentTracker — validate against new trial results
  7. Register model version in ModelRegistry with AUC and recall@k metrics
"""
)

# Register the link predictor
if has_registry:
    version = register_model(
        registry=registry,
        name="m5_gnn_link_predictor",
        model=link_model,
        metrics=[
            MetricSpec(name="final_link_auc", value=final_auc),
            MetricSpec(name="final_link_loss", value=link_losses[-1]),
            MetricSpec(name="rediscovery_auc", value=rediscovery_auc),
        ],
    )
    print(
        f"  Registered link_predictor: version={version.version}, auc={final_auc:.4f}"
    )


### Apply Checkpoint ────────────────────────────────────────────────


In [ ]:
assert rediscovery_auc > 0.5, "Rediscovery AUC should beat random"
print("\n--- Apply checkpoint passed --- knowledge graph completion demonstrated\n")


## REFLECTION


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — Link Prediction")
print("=" * 70)
print(
    f"""
  LINK PREDICTION WITH GNN ENCODER-DECODER:
  [x] Encoder: GCN layers produce node embeddings from features + structure
  [x] Decoder: dot-product similarity — score(i,j) = z_i^T z_j
  [x] Training: positive edges (real) vs negative edges (sampled non-edges)
  [x] AUC metric: {final_auc:.1%} — ranks real edges above non-edges
  [x] Rediscovery experiment: {rediscovery_auc:.1%} AUC on withheld edges
  [x] Visualised score distributions and similarity heatmaps

  LINK PREDICTION vs NODE CLASSIFICATION:
  - Node classification: "what IS this node?" (label prediction)
  - Link prediction: "should these nodes be CONNECTED?" (edge prediction)
  - Same GNN encoder; different decoder (classifier vs dot product)
  - Link prediction is the foundation of recommendation systems

  APPLICATIONS:
  - Knowledge graphs: discover new drug-disease interactions
  - Social networks: "people you may know"
  - Citation networks: "papers you should cite"
  - E-commerce: "products frequently bought together"

  Next: Exercise 6.5 — Architecture Comparison: systematic side-by-side
  evaluation of GCN vs GAT vs GraphSAGE on the same dataset...
"""
)

# Clean up
await conn.close()
